Create crosswalks between NextGen catchments and donor basins for all four domains

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import glob

In [ ]:
# define function to handle nested gages
def resolve_nested_gages(df: pd.DataFrame, nested_gages: str) -> pd.DataFrame:
    """Resolve nested gages by keeping only one gage per divide.

    For divides associated with multiple gages, keep the gage with
    the smallest or largest total drainage area.
    """
    id_col = "divide_id"

    # Find divides with multiple gages
    duplicates = df[df["divide_id"].duplicated(keep=False)]

    if duplicates.empty:
        return df

    # Compute drainage area for each gage (restricted to nested cases)
    gage_area = (
        duplicates.groupby("gage_id", as_index=False)[id_col]
        .sum()
        .rename(columns={id_col: "drainage_area"})
    )

    # Join back to duplicates
    nested = duplicates.merge(gage_area, on="gage_id", how="left")

    # Select either the min or max drainage area per divide
    if nested_gages == "inner":
        keep = nested.loc[nested.groupby("divide_id")["drainage_area"].idxmin()]
    elif nested_gages == "outer":
        keep = nested.loc[nested.groupby("divide_id")["drainage_area"].idxmax()]
    else:
        raise ValueError(
            f"Invalid value for 'nested_gages': {nested_gages}. "
            "Expected 'inner' or 'outer'."
        )

    # Keep all unique divides + the chosen nested gages
    keep = keep.drop(columns=["drainage_area"])
    result = pd.concat([df[~df["divide_id"].isin(duplicates["divide_id"])], keep])

    return result.reset_index(drop=True)


In [ ]:
# data directory for regionalization
outdir = Path("/home/yuqiong.liu/repos/nwm-region-mgr/data/inputs/cwt_divide_gage")
outdir.mkdir(exist_ok=True, parents=True)

# define domains
domains = {
    "conus": "CONUS",
    "ak": "Alaska",
    "hi": "Hawaii",
    "prvi": "Puerto_Rico",
    "gl": "Great_Lakes",
}

In [ ]:
# loop through domains to build crosswalk files between gage and catchment for calibration
ngage = ncats = 0
for domain1, domain in domains.items():
    # check if crosswalk file already exists, skip
    outfile = Path(outdir, "calib_gage_divide_" + domain1 + ".parquet")
    if outfile.exists():
        print(f"Crosswalk file already exists: {outfile}. Skip")
        continue

    # gather calibration basins and catchments for the domain
    dir1 = Path("/home/yuqiong.liu/work/data/gpkg_v2.2/", domain).resolve(strict=True)

    # identify all gpkg files
    files = glob.glob(f"{dir1}/*.gpkg")

    # loop through gpkg files to get divide_ids
    df_cats = pd.DataFrame()
    for f1 in files:
        # get divide_id and a few others attributes
        cats = gpd.read_file(f1, layer="divides")
        cols = ["divide_id", "toid", "areasqkm", "vpuid", "type"]
        cats = cats.reindex(columns=cols, fill_value=float("nan"))

        # get gage id
        cats["gage_id"] = (
            Path(f1)
            .name.replace("gages-", "")
            .replace("gauge_", "")
            .replace(".gpkg", "")
        )

        # move 'gage' to be the first column
        cats.insert(0, "gage_id", cats.pop("gage_id"))

        # add to the overall dataframe
        df_cats = pd.concat([df_cats, cats], ignore_index=True, axis=0)

    # resolve nested gages if any, keep the inner gage only
    df_cats = resolve_nested_gages(df_cats, nested_gages="inner")

    # save crosswalk to file for use in formulation regionalization later
    df_cats.to_parquet(outfile, index=False)

    print(
        f"There are {len(files)} calibration gages and {len(df_cats)} catchments in the {domain} domain"
    )

    ngage = ngage + len(files)
    ncats = ncats + len(df_cats)

print(f"\nTotal number of basins: {ngage}")
print(f"Total number of catchments: {ncats}")

There are 1533 calibration gages and 165171 catchments in the CONUS domain
There are 26 calibration gages and 8318 catchments in the Alaska domain
There are 31 calibration gages and 103 catchments in the Hawaii domain
There are 50 calibration gages and 545 catchments in the PuertoRico domain
There are 27 calibration gages and 2257 catchments in the GreatLakes domain

Total number of basins: 1667
Total number of catchments: 176394


In [5]:
# check if there are NWMv4 calibration basins with missing in the crosswalks
df_gages = pd.read_csv("/home/yuqiong.liu/work/data/gages_nwm4_calib_all.csv")
df_gages["domain"] = df_gages["domain"].str.replace(" ", "", regex=False)

for domain1, domain in domains.items():
    outfile = Path(outdir, "calib_gage_divide_" + domain1 + ".parquet")
    df1 = pd.read_parquet(outfile)
    gages1 = df1["gage_id"].to_list()

    gages_nwm4 = df_gages[df_gages["domain"] == domain]["gage_id"].to_list()
    gages_missed = [g1 for g1 in gages_nwm4 if g1 not in gages1]
    gages_extra = [g1 for g1 in gages1 if g1 not in gages_nwm4]
    if gages_extra:
        print(f"The following calibration basins are extra in {domain}: {gages_extra}")

    if gages_missed:
        print(
            f"The following calibration basins are missing in {domain}: {gages_missed}"
        )

The following calibration basins are missing in Alaska: ['15493000', '15056210']
The following calibration basins are extra in GreatLakes: ['02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '0

In [6]:
print(df_cats)

      gage_id  divide_id       toid   areasqkm  vpuid     type
0     02FD001  cat-21558  nex-21549  17.928201    NaN  network
1     02FD001  cat-21561  nex-21549   8.547044    NaN  network
2     02FD001  cat-21550  nex-21551  14.248898    NaN  network
3     02FD001  cat-21553  nex-21551  10.940986    NaN  network
4     02FD001  cat-21551  nex-21552  16.357638    NaN  network
...       ...        ...        ...        ...    ...      ...
2252  02ED003  cat-17668  nex-17669   9.952240    NaN  network
2253  02ED003  cat-17671  nex-17672   9.893820    NaN  network
2254  02ED003  cat-17673  nex-17674   9.865926    NaN  network
2255  02ED003  cat-17728  nex-17729  14.820945    NaN  network
2256  02ED003  cat-17782  nex-17783   9.686530    NaN  network

[2257 rows x 6 columns]


In [8]:
domain1 = "conus"
outfile = Path(outdir, "calib_gage_divide_" + domain1 + ".parquet")
df1 = pd.read_parquet(outfile)
print(df1)

         gage_id    divide_id         toid   areasqkm vpuid     type
0       02365470   cat-503034   nex-503035  16.588350   03W  network
1       02365470   cat-503032   nex-503033  11.973600   03W  network
2       02365470   cat-503033   nex-503034  10.245150   03W  network
3       02365470   cat-503035   nex-503036  16.407449   03W  network
4       02365470   cat-503036   nex-503037  21.387600   03W  network
...          ...          ...          ...        ...   ...      ...
165166  13148500  cat-3017163  nex-3017164   4.811401    17  network
165167  12210900  cat-3056955  nex-3056956   5.032801    17  network
165168  12210900  cat-3056959  nex-3056956   9.711000    17  network
165169  12210900  cat-3056956  nex-3056957   8.980200    17  network
165170  12210900  cat-3056957  nex-3056958  12.140550    17  network

[165171 rows x 6 columns]
